# Workstream 7: Rerank Diagnostics

candidate generation の問題と reranker の問題を分離します。exp014 系の LightGBM LTR summary、valid metric audit、objective comparison、過去 rerank 実験 artifact を読み込みます。


## Setup


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

try:
    from IPython.display import Image, Markdown, display
except Exception:
    Image = Markdown = display = None

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "EDA").exists():
            return candidate
    raise RuntimeError("Repository root was not found from the current working directory.")

ROOT = find_repo_root(Path.cwd())
EDA_DIR = ROOT / "EDA"
TABLE_DIR = EDA_DIR / "tables"
FIGURE_DIR = EDA_DIR / "figures"
SUMMARY_DIR = EDA_DIR / "summary"
EXPERIMENT_DIR = ROOT / "mcrs" / "experiments"
INFERENCE_DIR = ROOT / "exp" / "inference"

for directory in (TABLE_DIR, FIGURE_DIR, SUMMARY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"TABLE_DIR={TABLE_DIR}")


In [ ]:
def read_table(name: str, **kwargs) -> pd.DataFrame:
    path = TABLE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def read_csv_path(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def show_df(df: pd.DataFrame, n: int = 20) -> None:
    if df.empty:
        print("empty dataframe")
        return
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n).to_string(index=False))


def show_image(name: str) -> None:
    path = FIGURE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return
    if display is not None and Image is not None:
        display(Image(filename=str(path)))
    else:
        print(path)


def save_table(df: pd.DataFrame, name: str) -> Path | None:
    if df.empty:
        print(f"skip empty table: {name}")
        return None
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(ROOT)} ({len(df):,} rows)")
    return path


def barplot(df: pd.DataFrame, *, x: str, y: str, hue: str | None = None, title: str = "", rotate: int = 0, figsize=(10, 4)) -> None:
    if df.empty:
        print("skip empty plot")
        return
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.barplot(data=df, x=x, y=y, hue=hue, ax=ax)
    else:
        df.plot(kind="bar", x=x, y=y, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=rotate)
    fig.tight_layout()
    plt.show()


## Load Rerank Summary Artifacts


In [ ]:
summary_paths = [
    EXPERIMENT_DIR / "exp014_lightgbm_ltr_reranker" / "results" / "ablation_all9_equal_weight_topk_sampled" / "ablation_all9_equal_weight_topk_sampled_summary.csv",
    EXPERIMENT_DIR / "exp014b_lgbm_positive_group_filter" / "results" / "exp014b_positive_group_summary.csv",
    EXPERIMENT_DIR / "exp014c_valid_metric_audit" / "results" / "valid_metric_audit_summary.csv",
    EXPERIMENT_DIR / "exp014d_lgbm_objective_comparison" / "results" / "exp014d_objective_summary.csv",
]

frames = []
for path in summary_paths:
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        continue
    df = pd.read_csv(path)
    df.insert(0, "artifact", path.relative_to(ROOT).as_posix())
    df.insert(1, "family", path.parents[2].name)
    frames.append(df)
rerank_summary = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

save_table(rerank_summary, "rerank_bucket_metrics.csv")
show_df(rerank_summary, 100)


## All-Task Metric Gate


In [ ]:
metric_cols = [
    "family", "step", "candidate_topk", "objective", "valid_groups_total", "valid_groups_with_positive",
    "valid_ndcg@20", "valid_ndcg@20_positive_groups", "all_tasks_ndcg@20", "all_tasks_recall@20",
    "candidate_recall@20", "candidate_recall@100", "candidate_recall@500", "unique_top20",
]
if not rerank_summary.empty:
    cols = [c for c in metric_cols if c in rerank_summary.columns]
    metric_view = rerank_summary[cols].copy()
    if "all_tasks_ndcg@20" in metric_view.columns:
        metric_view = metric_view.sort_values("all_tasks_ndcg@20", ascending=False)
    show_df(metric_view, 100)

    if "all_tasks_ndcg@20" in rerank_summary.columns and "step" in rerank_summary.columns:
        plot_df = rerank_summary.dropna(subset=["all_tasks_ndcg@20"]).copy()
        fig, ax = plt.subplots(figsize=(12, 5))
        if sns is not None:
            sns.barplot(data=plot_df, x="step", y="all_tasks_ndcg@20", hue="family", ax=ax)
        ax.set_title("Rerank all-task nDCG@20 by step")
        ax.tick_params(axis="x", rotation=30)
        fig.tight_layout()
        plt.show()


## Objective And Source-Mix Comparison


In [ ]:
if not rerank_summary.empty:
    cols = [c for c in ["family", "objective", "step", "enabled_sources", "all_tasks_ndcg@20", "candidate_recall@500", "oracle_ndcg@20"] if c in rerank_summary.columns]
    source_view = rerank_summary[cols].copy() if cols else pd.DataFrame()
    show_df(source_view.sort_values("all_tasks_ndcg@20", ascending=False) if "all_tasks_ndcg@20" in source_view.columns else source_view, 100)

    if {"objective", "all_tasks_ndcg@20", "step"}.issubset(rerank_summary.columns):
        plot_df = rerank_summary.dropna(subset=["objective", "all_tasks_ndcg@20"])
        fig, ax = plt.subplots(figsize=(10, 4))
        if sns is not None:
            sns.barplot(data=plot_df, x="step", y="all_tasks_ndcg@20", hue="objective", ax=ax)
        ax.set_title("Objective comparison by step")
        fig.tight_layout()
        plt.show()


## Failure / Error Case Artifacts


In [ ]:
error_rows = []
for root in [EXPERIMENT_DIR / name for name in [
    "exp007_candidate_union_rerank_gate", "exp010_cross_modal_rerank", "exp011_qwen3_reranker_4b", "exp012_wide_candidate_generation", "exp014_lightgbm_ltr_reranker"
]]:
    if not root.exists():
        continue
    for path in sorted(root.rglob("*.jsonl")):
        try:
            count = sum(1 for _ in path.open())
        except UnicodeDecodeError:
            count = None
        error_rows.append({"family": root.name, "file": path.name, "rows": count, "path": path.relative_to(ROOT).as_posix()})
rerank_failure_cases = pd.DataFrame(error_rows)
save_table(rerank_failure_cases, "rerank_failure_cases.csv")
show_df(rerank_failure_cases, 100)


## Findings / Decisions / Next Actions

- Findings:
- Decisions:
- Next actions:
